In [ ]:
from langchain_groq import ChatGroq
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, trim_messages
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.runnables.history import RunnableWithMessageHistory
import os

class ImprovedPhysicsTeacherAgent:
    def __init__(self):
        # 🧠 LLM Setup
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

        # 🎓 Persona & Prompt with Memory Placeholder
        self.system_message = SystemMessage(content="You are an expert physics teacher.")
        self.prompt = ChatPromptTemplate.from_messages([
            self.system_message,
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}")
        ])

        # 🔗 Chain with Trimming (Windowed Memory)
        # Keeps only the last 5 messages to optimize token usage
        self.chain = self.prompt | self.llm
        self.store = {}

    def _get_history(self, session_id):
        if session_id not in self.store:
            self.store[session_id] = ChatMessageHistory()
        return self.store[session_id]

    def ask_question(self, question: str, session_id="student1"):
        history = self._get_history(session_id)
        
        # Trim history to last 5 messages before processing
        trimmed_history = trim_messages(
            history.messages,
            max_tokens=5, 
            strategy="last",
            token_counter=len
        )
        
        response = self.chain.invoke({
            "history": trimmed_history,
            "question": question
        })
        
        # Save new interaction
        history.add_user_message(question)
        history.add_ai_message(response.content)
        
        return response.content

# 🚀 Initialization
# agent = ImprovedPhysicsTeacherAgent()